In [2]:
import mysql.connector

## 13.3.1  数据库的创建与连接

In [79]:
#创建连接 
cnx= mysql.connector.connect(  
host="127.0.0.1",  
user="root",  
passwd="314159",
database='sakila')  
## 创建cursor对象
cursor = cnx.cursor()  

## 关闭游标和连接
cursor.close()
cnx.close()

In [80]:
#创建连接 
cnx= mysql.connector.connect(  
host="127.0.0.1",  
user="root",  
passwd="314159",
database='sakila')  
## 创建cursor对象
cursor = cnx.cursor(buffered=True)  

In [67]:
# cursor = cnx.cursor()  
# cursor.execute("SHOW TABLES;")
# tables = cursor.fetchall()  
# for table in tables:  
#     print(table[0])  # 输出表名  
cursor = cnx.cursor(buffered=False)  

## 13.3.2 通过游标执行数据库操作

In [100]:
# 执行非参数化查询
cursor.execute("SELECT * FROM film WHERE rating = 'G'")
# 执行参数化查询,使用%s占位符
query = "SELECT * FROM film WHERE rating = %s"
params = ("G",)
cursor.execute(query,params)
# 执行参数化查询，使用%(rating)s占位符
query = "SELECT * FROM film WHERE rating = %(rating)s"
params = {'rating':"G"}
cursor.execute(query,params)
# 非查询操作需提交事务
#cnx.commit()

## 13.3.3 通过游标获取查询结果

In [107]:
# 执行非参数化查询
cursor.execute("SELECT film_id, title FROM film WHERE rating = 'G'")
row = cursor.fetchone()
print(row)
row = cursor.fetchone()
print(row)

(2, 'ACE GOLDFINGER')
(4, 'AFFAIR PREJUDICE')


In [108]:
# 执行非参数化查询
cursor.execute("SELECT film_id, title FROM film WHERE rating = 'G'")
rows = cursor.fetchmany(3)
print(rows)
rows = cursor.fetchmany(3)
print(rows)

[(2, 'ACE GOLDFINGER'), (4, 'AFFAIR PREJUDICE'), (5, 'AFRICAN EGG')]
[(11, 'ALAMO VIDEOTAPE'), (22, 'AMISTAD MIDSUMMER')]


In [109]:
# 执行非参数化查询
cursor.execute("SELECT film_id, title FROM film WHERE rating = 'G' limit 5")
row = cursor.fetchone()
print(row)
rows = cursor.fetchmany(2)
print(rows)
rows = cursor.fetchall()
print(rows)

(2, 'ACE GOLDFINGER')
[(4, 'AFFAIR PREJUDICE'), (5, 'AFRICAN EGG')]
[(11, 'ALAMO VIDEOTAPE'), (22, 'AMISTAD MIDSUMMER')]


## 13.4.1 建立连接并创建数据库store

In [ ]:
import mysql.connector
# 连接数据库
cnx= mysql.connector.connect(host ="localhost",
                             user ="root",
                             passwd ="314159")

# 获得游标
cursor = cnx.cursor()
# 创建数据库，库名store
cursor.execute("CREATE DATABASE store")
cnx.commit()

cursor.close()
cnx.close()

## 13.4.2 建立数据表fruits并初始化数据

In [ ]:
# 创建数据表，此处以水果举例
cnx= mysql.connector.connect(host ="localhost",
                             user ="root",
                             passwd ="314159",
                            database='store')
cursor = cnx.cursor()
db_grocery = """CREATE TABLE fruits (
                   fruID  INT NOT NULL PRIMARY KEY,
                   fruName VARCHAR(50),
                   type VARCHAR(50),
                   fruPrice INT,
                   fruStock INT
                   )"""
try:
    cursor.execute(db_grocery)
except:
    print('failed')
 
# 初始化数据
sql = "INSERT INTO fruits  (fruID, fruName, type, fruPrice, fruStock                  ) values (%s, %s, %s, %s, %s)"
data1 = (1, '香蕉', '水果', 8, 140)
data2 = (2, '山竹', '水果', 12, 290)
data3 = (3, '樱桃', '水果', 30, 300)
data4 = (4, '凤梨', '水果', 17, 100)
data5 = (5, '石榴', '水果', 9, 210)
data_list = [data1, data2, data3, data4, data5]

cursor.executemany(sql,data_list)
cnx.commit()


## 13.4.3 添加新商品

In [116]:
def connect():
    cnx = mysql.connector.connect(host ="localhost",
            user ="root",
            passwd ="314159", database='store')# 获得游标

    # 获取操作游标
    cursor = cnx.cursor()
    return {"cnx": cnx, "cursor": cursor}


In [120]:
# 添加商品函数  
def add_goods():  
    # 获取操作游标  
    connection = connect()  
    cnx, cursor = connection['cnx'], connection['cursor']  
    try:  
        # 获取用户输入  
        fruID = int(input('请输入要添加的商品编号：'))  
        fruName = input('请输入要添加的商品名字：')  
        type = (input('请输入要添加的商品类型：'))  
        fruPrice = int(input('请输入商品价格：'))  
        fruStock = int(input('请输入商品库存：'))  
        # 插入数据  
        sql_insert = "INSERT INTO fruits (fruID, fruName, type, fruPrice, fruStock) VALUES (%s, %s, %s, %s, %s)"  
        data = (fruID, fruName, type, fruPrice, fruStock)  
        cursor.execute(sql_insert, data)  
        # 提交事务  
        cnx.commit()  
        print("商品添加成功！")  
    except Error as e:  
        # 捕获数据库异常  
        if e.error == 1062:# MySQL中主键冲突的错误代码是1062  
            print("错误：该商品编号已存在，请重新输入。")  
        else:  
            print("数据库错误：", e)  
    finally:  
        # 关闭连接  
        if cnx.is_connected():  
            cursor.close()  
            cnx.close()


In [121]:
add_goods()

请输入要添加的商品编号： 6
请输入要添加的商品名字： 草莓
请输入要添加的商品类型： 水果
请输入商品价格： 13
请输入商品库存： 300


商品添加成功！


## 13.4.4 下架商品

In [122]:
from mysql.connector import Error  
  
def delete_goods():  
    # 获取数据库连接  
    connection = connect()  
    cnx, cursor = connection['cnx'], connection['cursor']  
    try:  
        # 获取用户输入的商品编号  
        fruID = int(input('输入想要删除商品的编号：'))  
        # 执行删除操作  
        delete_query = "DELETE FROM fruits WHERE fruID = %s" 
        cursor.execute(delete_query, (fruID,))  
        # 提交事务  
        cnx.commit()  
        # 检查是否成功删除了记录  
        rowcount = cursor.rowcount  
        if rowcount == 1:  
            print('删除成功！')  
        elif rowcount == 0:  
            print('删除失败！可能这条商品记录不存在。')  
        else:  
            print(f'删除操作影响了{rowcount}条记录，这通常不发生。')  
    except ValueError:  
        print('输入错误：请输入一个有效的整数作为商品编号。')  
    except Error as e:  
        print(f'数据库错误：{e}')  
    finally:  
        # 关闭游标和连接  
        cursor.close()  
        cnx.close()  
delete_goods()

输入想要删除商品的编号： 6


删除成功！


## 13.4.5 查找商品操作

In [132]:
def g_by_id():
    # 获取操作游标
    connection = connect()
    cnx, cursor = connection['cnx'], connection['cursor']
    choice_id = int(input('请输入商品编号：'))
    cursor.execute('select * from fruits where fruID=%s',(choice_id,))
    fruits = cursor.fetchall()    #利用游标查找数据表，如果数据库中有此表捕获，没有报异常
    for j in fruits:
        print("============================================")
        print('---商品编号:{}    商品名称:{}   商品类型:{}  商品价格:{}  商品库存:{}---' .format(j[0], j[1], j[2], j[3], j[4]))
        print('查询成功')
        print("============================================")
    cnx.close()  
g_by_id()

请输入商品编号： 1


---商品编号:1    商品名称:香蕉   商品类型:水果  商品价格:8  商品库存:140---
查询成功


In [140]:
def g_by_name():
    # 获取操作游标
    connection = connect()
    cnx, cursor = connection['cnx'], connection['cursor']
    choose_name = input('请输入商品名称：')
    cursor.execute('select * from fruits where fruName =%s',(choose_name,))
    fruits = cursor.fetchall()   #利用游标查找数据表，如果数据库中有此表捕获，没有报异常
    for j in fruits:
        print("============================================")
        print('---商品编号:{}    商品名称:{}   商品类型:{}  商品价格:{}  商品库存:{}---' .format(j[0], j[1], j[2], j[3], j[4]))
        print('查询成功')
        print("============================================")
    cnx.close()
g_by_name()

请输入商品名称： 香蕉


---商品编号:1    商品名称:香蕉   商品类型:水果  商品价格:8  商品库存:140---
查询成功


## 13.4.6 修改商品

In [142]:
from mysql.connector import Error  
def update_goods_info():  
    # 连接到数据库  
    connection = connect()  
    cnx, cursor = connection['cnx'], connection['cursor']  
    try:  
        # 获取用户输入的商品编号  
        fruID = int(input('请输入要修改的商品编号：'))  
        # 获取用户输入的新商品信息  
        new_name = input('请输入新的商品名称：')  
        new_type = input('请输入新的商品类型：') 
        new_price = int(input('请输入新的商品价格：'))  
        new_stock = int(input('请输入新的商品库存：'))  
        # 构建更新查询语句  
        update_query = """  
        UPDATE fruits  
        SET fruName = %s, type = %s, fruPrice = %s, fruStock =%s
        WHERE fruID = %s  
        """  
        # 执行更新操作  
        cursor.execute(update_query, (new_name, new_type, new_price, new_stock, fruID))  
        cnx.commit()  
        # 检查是否成功更新了记录  
        rowcount = cursor.rowcount  
        if rowcount == 1:  
            print('商品信息修改成功！')  
        else:  
            print('商品信息修改失败！可能该商品编号不存在于数据库中。')  
    except ValueError:  
        print('输入错误：请确保所有输入都是有效的数据类型。')  
    except Error as e:  
        print(f'数据库错误：{e}')  
    finally:  
        # 关闭游标和连接  
        cursor.close()  
        cnx.close()
update_goods_info()

请输入要修改的商品编号： 5
请输入新的商品名称： 榴莲
请输入新的商品类型： 水果
请输入新的商品价格： 20
请输入新的商品库存： 190


商品信息修改成功！
